In [1]:
import requests
import time
import pandas as pd

API_KEY = "a520810c991739ab528f1a677413235d"

DISCOVER_URL = "https://api.themoviedb.org/3/discover/movie"

LANGUAGES = {
    "en": "English",
    "te": "Telugu",
    "hi": "Hindi",
    "kn": "Kannada",
    "ta": "Tamil"
}

session = requests.Session()
session.params = {
    "api_key": API_KEY,
    "sort_by":"popularity.desc"
                 }



def safe_request(url, retries=3):
    for i in range(retries):
        try:
            response = session.get(
                url,
                timeout=10,
                headers={"User-Agent": "Mozilla/5.0"}
            )
            response.raise_for_status()
            return response.json()
        except Exception as e:
            print(f" Retry {i+1}: {url}")
            time.sleep(1)

    return {}

def get_movie_extra(movie_id):
    try:
        
        credits_url = f"https://api.themoviedb.org/3/movie/{movie_id}/credits"
        credits = safe_request(credits_url)

        cast = [i["name"].replace(" ","").lower() for i in credits.get("cast", [])[:5]]

        director = ""
        for person in credits.get("crew", []):
            if person["job"] == "Director":
                director = person["name"].replace(" ","").lower()
                break

        keywords_url = f"https://api.themoviedb.org/3/movie/{movie_id}/keywords"
        keywords_data = safe_request(keywords_url)

        keywords = [i["name"] for i in keywords_data.get("keywords", [])]

        return cast, director, keywords

    except Exception as e:
        print("Extra fetch failed:", movie_id, e)
        return [], "", []


def fetch_movies(language_code, start_year=2000, end_year=2025, pages=2):
    movies = []

    for year in range(start_year, end_year + 1):
        print(f"\n {language_code.upper()} | Year {year}")

        for page in range(1, pages + 1):
            try:
                response = session.get(
                    DISCOVER_URL,
                    params={
                        "with_original_language": language_code,
                        "primary_release_year": year,
                        "page": page
                    },
                    timeout=10
                )

                response.raise_for_status()
                data = response.json()

                for m in data.get("results", []):
                    movie_id = m["id"]

                    print(f" Processing: {movie_id}")

                    cast, director, keywords = get_movie_extra(movie_id)

                    movies.append({
                        "tmdb_id": movie_id,
                        "title": m.get("title"),
                        "release_date": m.get("release_date"),
                        "release_year": year,
                        "language": language_code,
                        "genres": m.get("genre_ids", []),
                        "popularity": m.get("popularity"),
                        "vote_average": m.get("vote_average"),
                        "vote_count": m.get("vote_count"),
                        "overview": m.get("overview"),
                        "cast": " ".join(cast),
                        "director": director,
                        "keywords": " ".join(keywords)
                    })

                    time.sleep(0.3)

                print(f" Page {page} done ({len(data.get('results', []))} movies)")
                time.sleep(0.5)

            except Exception as e:
                print(f" Page error: {e}")
                time.sleep(2)

        
        time.sleep(1)

    return movies



In [3]:
language_dataframes = {}

for lang_code, lang_name in LANGUAGES.items():
    print(f"\n Fetching movies for {lang_name}")
    movies = fetch_movies(lang_code)
    df = pd.DataFrame(movies)
    language_dataframes[lang_code] = df

    print(f"{lang_name}: {len(df)} movies fetched")



 Fetching movies for English

 EN | Year 2000
 Processing: 4247
 Processing: 98
 Processing: 4234
 Processing: 11688
 Processing: 77
 Processing: 107
 Processing: 1359
 Processing: 8871
 Processing: 8358
 Processing: 9532
 Processing: 4327
 Processing: 1900
 Processing: 955
 Processing: 1443
 Processing: 2024
 Processing: 3981
 Processing: 2123
 Processing: 134
 Processing: 9741
 Processing: 7443
 Page 1 done (20 movies)
 Processing: 6282
 Processing: 10501
 Processing: 9679
 Processing: 392
 Processing: 7450
 Processing: 462
 Processing: 10567
 Processing: 9383
 Processing: 786
 Processing: 9600
 Processing: 1907
 Processing: 8584
 Processing: 2133
 Processing: 2085
 Processing: 10559
 Processing: 1493
 Processing: 2655
 Processing: 11978
 Processing: 641
 Processing: 1636
 Page 2 done (20 movies)

 EN | Year 2001
 Processing: 671
 Processing: 120
 Processing: 10950
 Processing: 14778
 Processing: 808
 Processing: 585
 Processing: 10878
 Processing: 4248
 Processing: 1734
 Processing

In [4]:
import os
BASE_DIR=r"C:\Users\aksha\OneDrive\Desktop\ARS(advanced recommendation system)\data\raw"
for lang_code, df in language_dataframes.items():
    file_path = os.path.join(BASE_DIR,f"tmdb_movies_{lang_code}.csv")
    df.to_csv(file_path, index=False)
    print("Saved",file_path)


Saved C:\Users\aksha\OneDrive\Desktop\ARS(advanced recommendation system)\data\raw\tmdb_movies_en.csv
Saved C:\Users\aksha\OneDrive\Desktop\ARS(advanced recommendation system)\data\raw\tmdb_movies_te.csv
Saved C:\Users\aksha\OneDrive\Desktop\ARS(advanced recommendation system)\data\raw\tmdb_movies_hi.csv
Saved C:\Users\aksha\OneDrive\Desktop\ARS(advanced recommendation system)\data\raw\tmdb_movies_kn.csv
Saved C:\Users\aksha\OneDrive\Desktop\ARS(advanced recommendation system)\data\raw\tmdb_movies_ta.csv


In [5]:
import os
print(os.getcwd())
BASE_DIR=r"C:\Users\aksha\OneDrive\Desktop\REBEM"


C:\Users\aksha


In [6]:
import requests 
url="https://api.themoviedb.org/3/movie/550?api_key=a520810c991739ab528f1a677413235d"
res=requests.get(url)
data=res.json()
print(data["title"])
print(data["overview"])

ConnectionError: ('Connection aborted.', ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None))

In [7]:
import requests
import time
import pandas as pd


API_KEY = "a520810c991739ab528f1a677413235d"

DISCOVER_URL = "https://api.themoviedb.org/3/discover/movie"

LANGUAGES = {
    "en": "English",
    "te": "Telugu",
    "hi": "Hindi",
    "kn": "Kannada",
    "ta": "Tamil"
}

session = requests.Session()
session.params = {
    "api_key": API_KEY,
    "sort_by": "popularity.desc"
}



def safe_request(url):
    try:
        response = session.get(
            url,
            timeout=10,
            headers={"User-Agent": "Mozilla/5.0"}
        )
        response.raise_for_status()
        return response.json()
    except Exception as e:
        print("Error:", e)
        return {}



def get_movie_extra(movie_id):

    credits_url = f"https://api.themoviedb.org/3/movie/{movie_id}/credits"
    credits = safe_request(credits_url)

    cast = [actor["name"].replace(" ","").lower() for actor in credits.get("cast", [])[:5]]

    director = ""
    for person in credits.get("crew", []):
        if person["job"] == "Director":
            director = person["name"].replace(" ","").lower()
            break

    
    keywords_url = f"https://api.themoviedb.org/3/movie/{movie_id}/keywords"
    keywords_data = safe_request(keywords_url)

    keywords = [k["name"] for k in keywords_data.get("keywords", [])]

    return cast, director, keywords


def fetch_2026_movies(language_code):

    movies = []

    print("Fetching 2026 movies (all languages)...")

    response = session.get(
        DISCOVER_URL,
        params={
            "with_original_language": language_code,
            "primary_release_year": 2026,
            "page": 1
        },
        timeout=10
    )

    data = response.json()

    for m in data.get("results", []):

        movie_id = m["id"]
        print("Processing:", m.get("title"))


        cast, director, keywords = get_movie_extra(movie_id)

        movies.append({
            "tmdb_id": movie_id,
            "title": m.get("title"),
            "release_date": m.get("release_date"),
            "language": m.get("original_language"),
            "genres": m.get("genre_ids"),
            "popularity": m.get("popularity"),
            "vote_average": m.get("vote_average"),
            "overview": m.get("overview"),

            # extra
            "cast": " ".join(cast),
            "director": director,
            "keywords": " ".join(keywords)
        })
        time.sleep(0.3)

    return movies



In [8]:
language_dataframes = {}

for lang_code, lang_name in LANGUAGES.items():
    print(f"\n Fetching movies for {lang_name}")
    movies = fetch_2026_movies(lang_code)
    df = pd.DataFrame(movies)
    language_dataframes[lang_code] = df

    print(f"{lang_name}: {len(df)} movies fetched")


 Fetching movies for English
Fetching 2026 movies (all languages)...
Processing: Peaky Blinders: The Immortal Man
Processing: Project Hail Mary
Processing: Shelter
Processing: Scream 7
Processing: Pretty Lethal
Processing: War Machine
Processing: Crime 101
Processing: Send Help
Processing: Greenland 2: Migration
Processing: GOAT
Processing: Whistle
Processing: "Wuthering Heights"
Processing: Hoppers
Processing: Mercy
Processing: Return to Silent Hill
Processing: The Wrecking Crew
Processing: The Deadly Little Mermaid
Processing: Hellfire
Processing: Do Not Enter
Processing: How to Make a Killing
English: 20 movies fetched

 Fetching movies for Telugu
Fetching 2026 movies (all languages)...
Processing: Vanaveera
Processing: The Rajasaab
Processing: Cheekatilo
Processing: Ustaad Bhagat Singh
Processing: Peddi
Processing: Vishnu Vinyasam
Processing: Dacoit
Processing: The Paradise
Processing: Amaravathiki Aahvanam
Processing: Funky
Processing: Fauzi
Processing: Biker
Processing: Bhartha 

In [9]:
import os
BASE_DIR=r"C:\Users\aksha\OneDrive\Desktop\ARS(advanced recommendation system)\data\raw"
for lang_code, df in language_dataframes.items():
    file_path = os.path.join(BASE_DIR,f"tmdb_movies_2026_{lang_code}.csv")
    df.to_csv(file_path, index=False)
    print("Saved",file_path)





Saved C:\Users\aksha\OneDrive\Desktop\ARS(advanced recommendation system)\data\raw\tmdb_movies_2026_en.csv
Saved C:\Users\aksha\OneDrive\Desktop\ARS(advanced recommendation system)\data\raw\tmdb_movies_2026_te.csv
Saved C:\Users\aksha\OneDrive\Desktop\ARS(advanced recommendation system)\data\raw\tmdb_movies_2026_hi.csv
Saved C:\Users\aksha\OneDrive\Desktop\ARS(advanced recommendation system)\data\raw\tmdb_movies_2026_kn.csv
Saved C:\Users\aksha\OneDrive\Desktop\ARS(advanced recommendation system)\data\raw\tmdb_movies_2026_ta.csv
